# EEG-shaped word representations → sentiment (EEG as a training signal)

EEG is used only during training; sentiment is predicted from text alone. Two versions, each with the same controls (no EEG objective, EEG targets shuffled across words, random targets). **EEG helps only if `eeg` beats `shuffled_eeg` and `random_targets`.**

* **A — static (minutes):** frozen LaBSE word vectors; the directions that predict word-level EEG are amplified; logistic regression on unseen ZuCo sentences, plus a transfer test on SST (no EEG).
* **B — brain-tuning (GPU):** the LoRA Qwen sentiment model gets an auxiliary head that predicts each word's EEG from its hidden state.

Needs the word-level EEG cache from the diagnostics notebook (`extract_word_eeg.py`).

## 1. Repository and dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

## 2. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
LABSE_DRIVE = f'{ARTIFACTS}/models/LaBSE'
ENCODER = LABSE_DRIVE if os.path.exists(f'{LABSE_DRIVE}/config.json') else 'sentence-transformers/LaBSE'
OUT = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_shaped'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(OUT, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
assert os.path.exists(WORD_EEG_DIR), 'run the word-EEG extraction first (diagnostics notebook)'
print('encoder:', ENCODER, '| device:', DEVICE)

## 3. Tests

In [ ]:
!python -m pytest -q tests/test_brainshaping.py

## 4. Version A — EEG-shaped static word vectors (ZuCo 5-fold + SST transfer)

In [ ]:
!python scripts/run_eeg_shaped_embeddings.py --word-eeg-dir "$WORD_EEG_DIR" --out-dir "$OUT/static" \
    --encoder "$ENCODER" --device $DEVICE --external sst5

## 5. Version B — brain-tuned LoRA LLM (GPU)

One fold to start (`FOLDS = 1`); raise to 5 later and finished folds are reused. One example per sentence, so this is much faster than the fusion run.

In [ ]:
FOLDS = 1
!python scripts/run_brain_tuning.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$OUT" --run-tag brain_tuning_v1 --folds $FOLDS

## 6. Reports

In [ ]:
from IPython.display import Markdown, display
for path in [f'{OUT}/static/eeg_shaped_embeddings.md', f'{OUT}/brain_tuning_v1/brain_tuning_results.md']:
    if os.path.exists(path):
        display(Markdown(open(path).read()))